# un-ai — KaniTTS ky (Kaggle T4 или Google Colab)

Kaggle отдаёт GPU только после **проверки телефона**. Если Accelerator = None — это не баг ноутбука.

**Kaggle:** аватар → Settings → Phone Verification → код из SMS → обновить страницу ноутбука → Session options → GPU T4 (или P100 / T4 x2) и Internet On.

**Если T4 так и нет — Colab (тот же файл):**
1. [colab.research.google.com](https://colab.research.google.com) → Upload notebook.
2. Вверху: **Среда выполнения** → **Сменить тип среды выполнения** → аппаратный ускоритель **T4 GPU** → **Сохранить**.
3. Справа сверху **Подключиться**. Затем **Среда выполнения** → **Выполнить все**. Скопируйте `KANI_BASE_URL` из последней ячейки.

Не запускайте отдельно `pip install kani-tts==1.0.1`: на Colab Python 3.13 pip часами перебирает `contourpy` / `ml-dtypes`.

Вкладку не закрывайте. Платный VPS не нужен.

In [ ]:
import os, torch
WORKDIR = "/kaggle/working" if os.path.isdir("/kaggle/working") else "/content"
print("WORKDIR", WORKDIR)
assert torch.cuda.is_available(), (
    "GPU нет. Kaggle: подтвердите телефон в Settings, затем Session options → GPU. "
    "Иначе Colab: Среда выполнения → Сменить тип среды выполнения → T4 GPU."
)
print("GPU:", torch.cuda.get_device_name(0), "VRAM", round(torch.cuda.get_device_properties(0).total_memory / 1024**3, 1), "GB")

In [ ]:
# Colab 3.13: kani-tts без --no-deps тянет nemo-toolkit[tts] и pip часами бэктрекает contourpy.
# NeMo нужен только как кодек; extra [tts] (matplotlib/datasets/optuna) для голоса не нужен.
%pip install -q fastapi uvicorn soundfile "numpy>=1.26" "scipy>=1.11" librosa omegaconf einops hydra-core
%pip install -q --no-deps kani-tts==1.0.1
%pip install -q --no-deps "nemo-toolkit==2.4.0"
%pip install -q -U "transformers==4.57.1"
import kani_tts
print("kani_tts OK", kani_tts.__file__)

In [ ]:
%%writefile server.py
from __future__ import annotations

import io
import os
import threading
from typing import Optional

import numpy as np
import soundfile as sf
from fastapi import FastAPI, HTTPException
from fastapi.responses import Response
from pydantic import BaseModel, Field

SAMPLE_RATE = 22050
MODEL_ID = os.getenv("KANI_MODEL", "nineninesix/kani-tts-400m-ky")
DEFAULT_VOICE = os.getenv("KANI_VOICE", "elina")

app = FastAPI(title="un-ai kani-tts")
_lock = threading.Lock()
_engine = None


class SpeechRequest(BaseModel):
    input: str
    model: Optional[str] = None
    voice: Optional[str] = Field(default=None)
    response_format: Optional[str] = "wav"


def _load_engine():
    global _engine
    if _engine is not None:
        return _engine
    from kani_tts import KaniTTS

    _engine = KaniTTS(MODEL_ID, show_info=True, suppress_logs=True)
    return _engine


def _to_wav_bytes(audio) -> bytes:
    if hasattr(audio, "detach"):
        audio = audio.detach().cpu().numpy()
    wave = np.asarray(audio).squeeze()
    if wave.ndim != 1:
        raise HTTPException(status_code=500, detail="unexpected audio shape")
    buf = io.BytesIO()
    sf.write(buf, wave, SAMPLE_RATE, format="WAV", subtype="PCM_16")
    return buf.getvalue()


def _speak(text: str, voice: str):
    engine = _load_engine()
    with _lock:
        try:
            return engine(text, speaker_id=voice)
        except TypeError:
            return engine(text)


@app.get("/health")
def health() -> dict[str, str]:
    try:
        import kani_tts  # noqa: F401
        installed = "yes"
    except ImportError:
        installed = "no"
    return {
        "status": "ok" if installed == "yes" else "missing_kani_tts",
        "service": "un-ai-kani",
        "model": MODEL_ID,
        "kani_tts": installed,
        "ready": "yes" if _engine is not None else "no",
    }


@app.post("/v1/audio/speech")
def speech(request: SpeechRequest) -> Response:
    text = request.input.strip()
    if not text:
        raise HTTPException(status_code=400, detail="empty input")
    voice = (request.voice or DEFAULT_VOICE).strip() or DEFAULT_VOICE
    try:
        audio, _generated = _speak(text, voice)
    except Exception as exc:
        raise HTTPException(status_code=500, detail=f"KaniTTS: {exc}") from exc
    return Response(content=_to_wav_bytes(audio), media_type="audio/wav")


In [ ]:
import os, re, subprocess, sys, time, urllib.request

import kani_tts  # noqa: F401 — без пакета туннель бесполезен

PORT = 8765
WORKDIR = "/kaggle/working" if os.path.isdir("/kaggle/working") else "/content"
os.environ.setdefault("KANI_MODEL", "nineninesix/kani-tts-400m-ky")
os.environ.setdefault("KANI_VOICE", "elina")
os.chdir(WORKDIR)

bin_path = f"{WORKDIR}/cloudflared"
if not os.path.exists(bin_path):
    print("скачиваю cloudflared…")
    urllib.request.urlretrieve(
        "https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64",
        bin_path,
    )
    os.chmod(bin_path, 0o755)

uvicorn_log = open(f"{WORKDIR}/uvicorn.log", "w")
tunnel_log_path = f"{WORKDIR}/tunnel.log"
tunnel_log = open(tunnel_log_path, "w")

uvicorn = subprocess.Popen(
    [sys.executable, "-m", "uvicorn", "server:app", "--host", "127.0.0.1", "--port", str(PORT)],
    cwd=WORKDIR,
    stdout=uvicorn_log,
    stderr=subprocess.STDOUT,
)
# HTTP/2: на Kaggle QUIC/UDP часто режут, из‑за этого туннель «молчит».
tunnel = subprocess.Popen(
    [bin_path, "tunnel", "--url", f"http://127.0.0.1:{PORT}", "--protocol", "http2", "--no-autoupdate"],
    stdout=tunnel_log,
    stderr=subprocess.STDOUT,
)

url = None
deadline = time.time() + 90
pattern = re.compile(r"https://[a-z0-9-]+\.trycloudflare\.com")
print("жду URL туннеля (до 90 сек)…")
while time.time() < deadline and url is None:
    time.sleep(2)
    text = open(tunnel_log_path, encoding="utf-8", errors="replace").read()
    found = pattern.search(text)
    if found:
        url = found.group(0)
    if uvicorn.poll() is not None:
        print("uvicorn упал:\n", open(f"{WORKDIR}/uvicorn.log", encoding="utf-8", errors="replace").read()[-4000:])
        break
    if tunnel.poll() is not None and url is None:
        print("cloudflared вышел:\n", text[-4000:])
        break

if not url:
    print("--- cloudflared.log ---")
    print(open(tunnel_log_path, encoding="utf-8", errors="replace").read()[-4000:])
    raise RuntimeError(
        "Туннель не поднялся. Нужны Settings → Internet: On и GPU T4. "
        "Пришлите хвост cloudflared.log из вывода выше."
    )

print("\n--- вставьте в .env на своём ПК ---")
print(f"KANI_BASE_URL={url}/v1")
print("KANI_VOICE=elina")
print("-----------------------------------")
print("Ноутбук не закрывайте. Первая фраза на ky качает веса (1–3 мин).")

try:
    while True:
        time.sleep(120)
        if uvicorn.poll() is not None or tunnel.poll() is not None:
            print("процесс упал, перезапустите ячейку")
            print(open(tunnel_log_path, encoding="utf-8", errors="replace").read()[-2000:])
            break
except KeyboardInterrupt:
    uvicorn.terminate()
    tunnel.terminate()